# Chapter 15: MLOps & Model Deployment
## Notebook 03 — Advanced MLOps

Production models fail differently from research models. This notebook covers what happens **after** you ship: monitoring drift, releasing safely, observing the system, scaling, and designing the capstone.

### What you'll learn

| Topic | Section |
|-------|--------|
| Data drift: PSI, KS test, prediction drift | §1–2 |
| Evidently sketch with a NumPy fallback | §3 |
| A/B testing and canary deploys (traffic-splitter simulation) | §4 |
| Observability: structured logs, metrics, tracing | §5 |
| Scaling, autoscaling, and cost trade-offs | §6 |
| Capstone design | §7 |

**Estimated time:** 2.5 hours

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from monitoring import psi, ks_stat, LatencyTracker, DriftDetector, Logger

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)
print('imports OK')

---
## 1. Data Drift: When the World Changes Under Your Model

A model trained on yesterday's distribution serves tomorrow's traffic. **Data drift** is when the input distribution shifts; **concept drift** is when the input → output relationship shifts. Both degrade quality silently — the model still returns predictions, just worse ones.

Two classic detectors:

- **PSI** (Population Stability Index) — bin-based divergence between reference and current distributions.
- **KS test** — compares the empirical CDFs; non-parametric, no binning needed.

Below we exercise both on synthetic shifted data.

In [ ]:
rng = np.random.default_rng(42)
ref = rng.normal(loc=0.0, scale=1.0, size=2000)         # reference (training data)
no_shift = rng.normal(loc=0.0, scale=1.0, size=2000)    # same distribution
mild_shift = rng.normal(loc=0.3, scale=1.0, size=2000)  # small mean shift
big_shift  = rng.normal(loc=1.0, scale=1.5, size=2000)  # larger shift

print(f'PSI reference vs no-shift  : {psi(ref, no_shift):.4f}')
print(f'PSI reference vs mild-shift: {psi(ref, mild_shift):.4f}')
print(f'PSI reference vs big-shift : {psi(ref, big_shift):.4f}')

print()
for name, cur in [('no-shift', no_shift), ('mild-shift', mild_shift), ('big-shift', big_shift)]:
    k = ks_stat(ref, cur)
    print(f'KS reference vs {name:<10}: D={k["statistic"]:.4f}, p={k["pvalue"]:.4g}')

**How to read these numbers**

| PSI | Interpretation |
|-----|----------------|
| < 0.10 | No significant change |
| 0.10 – 0.25 | Moderate shift — investigate |
| > 0.25 | Significant shift — alert |

| KS p-value | Interpretation |
|------------|----------------|
| > 0.05 | Cannot reject "same distribution" |
| < 0.05 | Distributions are statistically different |

PSI is more interpretable; KS is a tighter statistical test. Most teams use both.

In [ ]:
# Visualize the shift.
fig, ax = plt.subplots()
ax.hist(ref, bins=40, alpha=0.5, label='reference', density=True)
ax.hist(big_shift, bins=40, alpha=0.5, label='current (shifted)', density=True)
ax.set_title('Reference vs current — feature distribution drift')
ax.legend()
plt.show()

---
## 2. Drift on the Chapter's Datasets

`datasets/reference_data.csv` is a snapshot taken at training time. `datasets/current_data.csv` is the live distribution — `feature_a` is intentionally shifted upward. Run the orchestrator to produce a per-feature drift report.

In [ ]:
ref_df = pd.read_csv('../datasets/reference_data.csv')
cur_df = pd.read_csv('../datasets/current_data.csv')
print('reference shape:', ref_df.shape)
print('current   shape:', cur_df.shape)
print()
print('reference means:'); print(ref_df.mean().round(3))
print('current   means:'); print(cur_df.mean().round(3))

In [ ]:
detector = DriftDetector(psi_warn=0.10, psi_alert=0.25, ks_pvalue=0.05)
report = detector.detect(
    reference={c: ref_df[c].values for c in ref_df.columns},
    current  ={c: cur_df[c].values for c in cur_df.columns},
)
print(json.dumps(report, indent=2))

**Prediction drift** — even when input drift is small, the *output* distribution can shift (e.g., the share of `class=1` predictions). Track it the same way: PSI between predictions in a baseline window and the current window.

In [ ]:
# Synthetic prediction-drift example.
preds_ref = rng.binomial(1, 0.30, size=2000)
preds_now = rng.binomial(1, 0.42, size=2000)  # share of class=1 went 30% -> 42%
print('PSI on predictions:', round(psi(preds_ref, preds_now, bins=2), 4))
print('class=1 share - ref :', preds_ref.mean().round(3))
print('class=1 share - now :', preds_now.mean().round(3))

---
## 3. Evidently (with a NumPy Fallback)

[Evidently](https://github.com/evidentlyai/evidently) is a popular Python library for drift dashboards and reports. It isn't installed in this environment — we wrap the import in `try/except` and fall back to our own NumPy implementation.

In [ ]:
try:
    from evidently.report import Report  # type: ignore
    from evidently.metric_preset import DataDriftPreset  # type: ignore
    USE_EVIDENTLY = True
    print('evidently installed — full preset available')
except ImportError:
    USE_EVIDENTLY = False
    print('evidently not installed — using NumPy fallback (pip install evidently to upgrade)')

def evidently_or_fallback(reference: pd.DataFrame, current: pd.DataFrame) -> dict:
    if USE_EVIDENTLY:
        report = Report(metrics=[DataDriftPreset()])
        report.run(reference_data=reference, current_data=current)
        return report.as_dict()
    # Fallback: just call our DriftDetector
    return DriftDetector().detect(
        {c: reference[c].values for c in reference.columns},
        {c: current[c].values   for c in current.columns},
    )

result = evidently_or_fallback(ref_df, cur_df)
print(json.dumps(result, indent=2)[:800], '...')

---
## 4. A/B Tests and Canary Deploys

Two release strategies for new models:

- **A/B test** — split traffic 50/50 between v_A and v_B; compare a business metric over a fixed period.
- **Canary** — send a small fraction (1–10%) of traffic to the new model; ramp up if no regression.

Both reduce blast radius. Below we simulate a canary splitter: 10% of traffic goes to the candidate; the rest stays on the current Production model. We log per-version error rates and roll back automatically if the candidate degrades.

In [ ]:
class CanarySplitter:
    """Deterministic traffic splitter using a hash of the request id."""
    def __init__(self, candidate_share: float, seed: int = 0):
        assert 0 <= candidate_share <= 1
        self.share = candidate_share
        self.seed = seed
    def route(self, request_id: str) -> str:
        # Map id -> [0,1) deterministically; same id always routes the same way.
        h = abs(hash((self.seed, request_id))) % 10_000
        return 'candidate' if (h / 10_000) < self.share else 'production'

# Simulate 1000 requests with 10% canary share. Candidate has a slightly
# higher error rate (8%) than production (3%) — should trigger rollback.
splitter = CanarySplitter(candidate_share=0.10, seed=42)
counts = {'production': 0, 'candidate': 0}
errors = {'production': 0, 'candidate': 0}
for i in range(1000):
    rid = f'req-{i:05d}'
    route = splitter.route(rid)
    counts[route] += 1
    err_rate = 0.08 if route == 'candidate' else 0.03
    if rng.random() < err_rate:
        errors[route] += 1

for route in counts:
    rate = errors[route] / max(counts[route], 1)
    print(f'  {route:<11}  n={counts[route]:4}  errors={errors[route]:3}  err_rate={rate:.3f}')

# Rollback policy
prod_rate = errors['production'] / max(counts['production'], 1)
cand_rate = errors['candidate'] / max(counts['candidate'], 1)
if cand_rate > prod_rate * 1.5:
    print('\nROLLBACK: candidate error rate > 1.5x production')
else:
    print('\nOK to ramp up')

**Canary checklist:**

- Route deterministically (hash of user id) so the same user sees the same model.
- Compare the **same metric** between arms during the same window.
- Ramp **slowly**: 1% → 5% → 25% → 50% → 100% with a soak time at each step.
- Have a single command to roll back to the prior Production version.
- Measure both **technical** (latency, errors) and **product** (conversion, click-through) metrics.

---
## 5. Observability: Logs, Metrics, Tracing

Three pillars:

| Pillar | Best for | Example |
|--------|----------|---------|
| **Logs** | Discrete events with detail | Single bad prediction with the input |
| **Metrics** | Numeric aggregates over time | p99 latency, requests/sec |
| **Tracing** | Per-request cause-and-effect across services | preprocess took 80ms, inference 12ms, postprocess 3ms |

Below we wire up structured JSON logs (with our `Logger`), a `LatencyTracker`, and sketch a Prometheus-style metrics emitter.

In [ ]:
log_path = Path('../logs/predictions.jsonl')
if log_path.exists():
    log_path.unlink()
logger = Logger(log_path)
latency = LatencyTracker(window=500)

# Simulate 200 predictions
for i in range(200):
    t = float(rng.normal(50, 20))   # ms
    if rng.random() < 0.02:
        t = float(rng.uniform(200, 500))  # tail latency event
    latency.record(t)
    logger.log(
        'prediction',
        request_id=f'req-{i:05d}',
        feature_a=float(rng.normal()),
        prediction=int(rng.integers(0, 2)),
        latency_ms=round(t, 2),
        model_version='0.1.0',
    )

print('latency report:', latency.report())
print('\nfirst log line:')
print(logger.read_all()[0])

In [ ]:
# Sketch a Prometheus-style metrics emitter (no real client required).
try:
    from prometheus_client import Counter, Histogram  # type: ignore
    USE_PROM = True
    print('prometheus_client installed')
except ImportError:
    USE_PROM = False
    print('prometheus_client not installed — sketching shape')

EXPORT = '''
# HELP predictions_total Total predictions served.
# TYPE predictions_total counter
predictions_total{model="demo-classifier",version="0.1.0"} 200

# HELP predict_latency_ms Latency of /predict in ms.
# TYPE predict_latency_ms histogram
predict_latency_ms_bucket{le="50"} 122
predict_latency_ms_bucket{le="100"} 188
predict_latency_ms_bucket{le="200"} 196
predict_latency_ms_bucket{le="+Inf"} 200
predict_latency_ms_sum 12030
predict_latency_ms_count 200
'''.strip()
print(EXPORT)

**Tracing** is conceptually straightforward: every request carries a `trace_id`, and each component emits spans with start/end times tagged with `trace_id` + `span_id`. OpenTelemetry is the industry standard. For our purposes, the structured log already carries `request_id`; that's the seed of a trace.

---
## 6. Scaling and Cost

Two axes:

- **Vertical** — bigger machine: more RAM, more cores, GPU. Easy until it isn't.
- **Horizontal** — more replicas behind a load balancer. The default for stateless services.

**Autoscaling** — add replicas when CPU > 70% (or queue depth > N) for a sustained window; remove them when load drops. Kubernetes' HPA is the standard.

**Cost levers** for ML serving:
1. **Quantization / distillation** — smaller model, lower latency, often only marginal accuracy loss.
2. **Batching** — improves throughput per dollar (already covered in NB 01).
3. **Spot / preemptible nodes** — cheap, but you must handle eviction.
4. **Right-sized models** — a logistic regression that ships in 50KB beats a 7B-parameter LLM for tabular tasks.

A useful back-of-envelope:

```
cost_per_request ≈ (instance_$_per_hour × instance_hours) / requests_served
```

Halving latency *or* doubling batch size both halve cost-per-request — choose whichever is cheaper to engineer.

In [ ]:
# Quick cost calculator
def cost_per_million_requests(instance_per_hour: float, rps: float) -> float:
    """$ per million requests assuming 100% utilization."""
    seconds_per_million = 1_000_000 / max(rps, 1e-9)
    hours = seconds_per_million / 3600
    return instance_per_hour * hours

scenarios = [
    ('small CPU box',   0.10, 200),    # $0.10/hr, 200 rps
    ('big GPU box',     2.50, 5000),   # $2.50/hr, 5000 rps
    ('serverless cold', 0.30, 50),     # cold path
]
for name, price, rps in scenarios:
    print(f'  {name:<18}  ${cost_per_million_requests(price, rps):6.2f} per 1M requests')

---
## 7. Capstone Design

Your capstone for this chapter: take **any model from a previous chapter** and put it into production — for real, end to end.

**Deliverables:**

1. **Repo structure** — `scripts/`, `tests/`, `notebooks/`, `Dockerfile`, `requirements.txt`, `.github/workflows/ci.yml`
2. **Pipeline** — sklearn `Pipeline` (or framework equivalent) that handles all preprocessing inside the artifact
3. **Service** — FastAPI with `/predict`, `/predict/batch`, `/health`, `/version`; tested via `TestClient`
4. **Registry** — at least 2 versions registered, one in Production
5. **CI** — lint + tests + smoke train + eval gates + auto-register on `main`
6. **Monitoring** — structured logs of every prediction, drift report comparing reference vs current windows, latency percentiles
7. **Runbook** — a short doc covering: how to deploy, how to rollback, what to do when drift fires

**Stretch:**

- Implement a canary splitter and run a synthetic A/B test
- Add a Prometheus metrics endpoint
- Containerize and run the image locally (`docker run`)
- Write an incident post-mortem for a deliberate failure injection

**Done means**: someone else on your team can clone the repo, read the runbook, and ship a new version safely without asking you a question.

---
## 8. Key Takeaways

- **Drift detection** combines a binned divergence (PSI) and a non-parametric test (KS); track input *and* prediction drift.
- **Canary deploys** ramp slowly, route deterministically, and compare like-for-like metrics.
- **Observability** = logs (events) + metrics (aggregates) + tracing (causality). Each request needs an id.
- **Scaling**: horizontal first, autoscaling on CPU or queue depth; quantize/distill for cost; the simplest model that meets the bar wins.
- **Operate models** like real services: runbook, rollback, on-call, post-mortems.

---

## What's Next

This chapter completes the **Practitioner Track**. You can now train, package, deploy, monitor, and operate a model end to end.

The **Advanced Track** (Chapter 16+) takes you into agentic systems, evaluation at scale, multimodal models, and alignment — the open frontier of AI.

---
*Generated by Berta AI*